# Catan Game Logs Data Pipeline
This notebook reads raw game log HTML fragments and extracts structured data using Polars.

In [6]:
import polars as pl
from bs4 import BeautifulSoup
import re
import os
import glob
from tqdm import tqdm


## Parser Functions

In [7]:
def parse_log_line(html_string):
    soup = BeautifulSoup(html_string, 'html.parser')
    feed_msg = soup.find('div', class_='feedMessage-O8TLknGe')
    if not feed_msg:
        return None
    
    msg_part = feed_msg.find('span', class_='messagePart-XeUsOgLX')
    if not msg_part:
        return None
        
    text = msg_part.get_text()
    
    # Check if HR
    if msg_part.find('hr'):
        return {'action_type': 'end_of_turn'}
        
    action_data = {'raw_text': text}
    
    # Simple regex matches
    if 'placed a Settlement' in text:
        action_data['action_type'] = 'placed_settlement'
        action_data['player'] = msg_part.find('span').text
    elif 'placed a Road' in text:
        action_data['action_type'] = 'placed_road'
        action_data['player'] = msg_part.find('span').text
    elif 'built a Settlement' in text:
        action_data['action_type'] = 'built_settlement'
        action_data['player'] = msg_part.find('span').text
    elif 'built a City' in text:
        action_data['action_type'] = 'built_city'
        action_data['player'] = msg_part.find('span').text
    elif 'built a Road' in text:
        action_data['action_type'] = 'built_road'
        action_data['player'] = msg_part.find('span').text
    elif 'received starting resources' in text:
        action_data['action_type'] = 'starting_resources'
        action_data['player'] = msg_part.find('span').text
        action_data['resources'] = [img.get('alt') for img in msg_part.find_all('img') if img.get('alt')]
    elif 'rolled' in text:
        action_data['action_type'] = 'rolled_dice'
        action_data['player'] = msg_part.find('span').text
        dice = [int(img.get('alt').split('_')[1]) for img in msg_part.find_all('img') if 'dice_' in img.get('alt', '')]
        action_data['roll_total'] = sum(dice) if dice else None
    elif 'moved Robber' in text:
         action_data['action_type'] = 'moved_robber'
         action_data['player'] = msg_part.find('span').text
    elif 'stole' in text and 'from' in text:
        if text.startswith('You stole'):
            action_data['action_type'] = 'stole'
            action_data['player'] = 'You'
            action_data['target'] = msg_part.find_all('span')[-1].text
        elif 'stole 2' in text or 'stole 1' in text or 'stole' in text:
            # Need to distinguish between normal steal and monopoly steal
            if 'stole' in text and len(msg_part.find_all('span')) >= 2:
                 action_data['action_type'] = 'stole'
                 action_data['player'] = msg_part.find_all('span')[0].text
                 action_data['target'] = msg_part.find_all('span')[-1].text
            else:
                 action_data['action_type'] = 'monopoly_steal'
                 action_data['player'] = msg_part.find_all('span')[0].text
                 action_data['stolen_resource'] = msg_part.find('img').get('alt') if msg_part.find('img') else None
    elif 'wants to give' in text and 'for' in text:
        action_data['action_type'] = 'proposed_trade'
        action_data['player'] = msg_part.find('span').text
    elif 'Friendly Robber is active' in text:
        action_data['action_type'] = 'friendly_robber'
    elif 'blocked by the Robber' in text:
        action_data['action_type'] = 'robber_blocked'
    elif 'got' in text and not 'gave' in text and not 'discarded' in text:
        # Avoid matching 'gave X and got Y'
        action_data['action_type'] = 'got_resource'
        if msg_part.find('span'):
            action_data['player'] = msg_part.find('span').text
    elif 'proposed counter offer' in text:
        action_data['action_type'] = 'counter_offer'
        spans = msg_part.find_all('span')
        if len(spans) >= 2:
            action_data['player'] = spans[0].text
            action_data['target'] = spans[1].text
    elif 'gave' in text and 'and got' in text and 'from' in text:
         action_data['action_type'] = 'completed_trade'
         spans = msg_part.find_all('span')
         if len(spans) >= 2:
            action_data['player'] = spans[0].text
            action_data['target'] = spans[-1].text
    elif 'gave bank' in text and 'and took' in text:
         action_data['action_type'] = 'bank_trade'
         action_data['player'] = msg_part.find('span').text
    elif 'bought' in text and 'Development Card' in str(msg_part):
         action_data['action_type'] = 'bought_dev_card'
         action_data['player'] = msg_part.find('span').text
    elif 'discarded' in text:
         action_data['action_type'] = 'discarded'
         action_data['player'] = msg_part.find('span').text
         action_data['discard_count'] = len(msg_part.find_all('img'))
    elif 'used' in text and 'Knight' in str(feed_msg):
        action_data['action_type'] = 'played_knight'
        action_data['player'] = msg_part.find('span').text
    elif 'used' in text and 'Monopoly' in str(feed_msg):
        action_data['action_type'] = 'played_monopoly'
        action_data['player'] = msg_part.find('span').text
    elif 'received Longest Road' in text:
        action_data['action_type'] = 'longest_road'
        action_data['player'] = msg_part.find('span').text
    elif 'received Largest Army' in text:
        action_data['action_type'] = 'largest_army'
        action_data['player'] = msg_part.find('span').text
    elif 'Longest Road' in text and 'passed from' in text:
        action_data['action_type'] = 'longest_road_transferred'
    elif 'selecting cards to discard' in text:
        action_data['action_type'] = 'bot_discarding'
    elif 'No player to steal from' in text:
        action_data['action_type'] = 'no_steal_target'
    elif 'won the game' in text:
        action_data['action_type'] = 'win'
        action_data['player'] = msg_part.find('span').text
    elif 'blocked trading with' in text:
        action_data['action_type'] = 'blocked_trading'
    else:
        action_data['action_type'] = 'unknown'
        
    return action_data


## Load and Process Data

In [8]:
def process_game_file(filepath):
    game_id = os.path.basename(filepath).split('_')[2].split('.')[0]
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split the content into divs roughly (each div is separated by double newlines usually)
    # Using BeautifulSoup directly on the whole file or splitting by <div data-index
    divs = re.split(r'\n(?=<div data-index)', content)
    
    actions = []
    turn_counter = 0
    trade_proposed_this_turn = False
    
    for div_html in divs:
        if not div_html.strip():
            continue
            
        parsed = parse_log_line(div_html)
        if parsed:
            parsed['game_id'] = game_id
            parsed['turn_number'] = turn_counter
            
            if parsed['action_type'] == 'end_of_turn':
                turn_counter += 1
                trade_proposed_this_turn = False
                
            if parsed['action_type'] == 'proposed_trade':
                if not trade_proposed_this_turn:
                    parsed['first_trade_of_turn'] = True
                    trade_proposed_this_turn = True
            
            actions.append(parsed)
            
    return actions

all_actions = []
data_dir = '../../data/raw_divs'
file_paths = glob.glob(os.path.join(data_dir, '*.txt'))

for fp in tqdm(file_paths):
    all_actions.extend(process_game_file(fp))
    
# Convert to Polars DataFrame
df = pl.DataFrame(all_actions)


100%|██████████| 27/27 [00:01<00:00, 16.33it/s]


## Calculate Stats per Game

In [9]:
# View some of the parsed data
print(df.head())
print("Unique action types extracted:", df['action_type'].unique().to_list())


shape: (5, 9)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬────────┬───────────┐
│ raw_text   ┆ action_typ ┆ game_id   ┆ turn_numb ┆ … ┆ resources ┆ roll_tota ┆ target ┆ first_tra │
│ ---        ┆ e          ┆ ---       ┆ er        ┆   ┆ ---       ┆ l         ┆ ---    ┆ de_of_tur │
│ str        ┆ ---        ┆ str       ┆ ---       ┆   ┆ list[str] ┆ ---       ┆ str    ┆ n         │
│            ┆ str        ┆           ┆ i64       ┆   ┆           ┆ i64       ┆        ┆ ---       │
│            ┆            ┆           ┆           ┆   ┆           ┆           ┆        ┆ bool      │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪════════╪═══════════╡
│ Happy      ┆ unknown    ┆ 192053656 ┆ 0         ┆ … ┆ null      ┆ null      ┆ null   ┆ null      │
│ settling!  ┆            ┆           ┆           ┆   ┆           ┆           ┆        ┆           │
│ Learn how  ┆            ┆           ┆           ┆   ┆           ┆          

In [10]:
def calculate_game_stats(df_game, game_id):
    # Get unique players
    players = df_game.filter(pl.col('player').is_not_null())['player'].unique().to_list()
    # Remove 'You' if it exists and map to the actual player if needed, or just exclude 'You' for general stats
    players = [p for p in players if p != 'You']
    
    stats = []
    for player in players:
        p_df = df_game.filter(pl.col('player') == player)
        
        stat = {
            'player': player,
            'settlements_built': p_df.filter(pl.col('action_type').is_in(['placed_settlement', 'built_settlement'])).height,
            'cities_built': p_df.filter(pl.col('action_type') == 'built_city').height,
            'roads_built': p_df.filter(pl.col('action_type').is_in(['placed_road', 'built_road'])).height,
            'bank_trades': p_df.filter(pl.col('action_type') == 'bank_trade').height,
            'trades_proposed': p_df.filter(pl.col('action_type') == 'proposed_trade').height,
            'trades_completed': p_df.filter(pl.col('action_type') == 'completed_trade').height,
            'dev_cards_bought': p_df.filter(pl.col('action_type') == 'bought_dev_card').height,
            'times_discarded': p_df.filter(pl.col('action_type') == 'discarded').height,
            'counter_offers': p_df.filter(pl.col('action_type') == 'counter_offer').height,
            'longest_road_received': p_df.filter(pl.col('action_type') == 'longest_road').height,
            'largest_army_received': p_df.filter(pl.col('action_type') == 'largest_army').height,
            'unique_players_stolen_from': p_df.filter(pl.col('action_type') == 'stole')['target'].n_unique() if 'target' in p_df.columns else 0,
            # Add more specific stat counts following stats_idea.txt
        }
        stats.append(stat)
        
    return pl.DataFrame(stats)

# Iterate through each unique game_id, calculate stats, and export to CSV
out_dir = '../../data/game_log_dfs'
os.makedirs(out_dir, exist_ok=True)

unique_games = df['game_id'].unique().to_list()
for game_id in tqdm(unique_games, desc="Exporting Game Stats"):
    df_game = df.filter(pl.col('game_id') == game_id)
    game_stats = calculate_game_stats(df_game, game_id)
    
    out_path = os.path.join(out_dir, f"game_{game_id}_stats.csv")
    game_stats.write_csv(out_path)

print(f"Exported stats for {len(unique_games)} games to {out_dir}")


Exporting Game Stats: 100%|██████████| 27/27 [00:00<00:00, 107.98it/s]

Exported stats for 27 games to ../../data/game_log_dfs
